# Lab 9 · BPR-MF

The runnable companion of the lab page, `docs/labs/09-bpr-mf.md` (also on the docs site). The page explains each step,
asks the questions and holds the hints and solutions; this notebook has the code.

- Run the setup cell first, every time the kernel starts (after a crash: **Restart**, then setup again).
- Cells that fit models take seconds to minutes on MovieLens. Run them where you train: on the rented box (VS Code's
  Remote - SSH extension opens this notebook there), or on the laptop if you allow training there.
- Cells that only read stored results (the baseline, `lab.compare`, `lab.segments`) are fine anywhere, once the box's
  results are fetched (`./scripts/fetch_lab_results.sh`).
- Save the notebook **without outputs** before you commit (the **...** menu → Clear All Outputs).

In [ ]:
from recbench import lab

lab.setup()
DATASET = "movielens-25m"
METHOD = "bpr_mf"

## Your starting point

The lab baseline on every dataset (stored results; nothing is trained here).

In [ ]:
lab.baseline(METHOD)

## Level 1: reproduce and read

### 1.1 Reproduce the baseline's best setting

Fit the baseline's best setting on the validation fold, in this notebook. The NDCG@10 must equal the baseline's validation score.

In [ ]:
best = lab.baseline(METHOD, DATASET)["best_settings"]
data, split = lab.load(DATASET)
model = lab.fit(METHOD, data, **best)
result = lab.evaluate(model, data, split)
print(result)
print("the baseline's best trial:", lab.baseline(METHOD, DATASET)["validation"])

### 1.2 Read the code

The questions are on the lab page; answer them with the code open next to this notebook.

### 1.3 Look inside

In [ ]:
item = int(data.item_pop.argmax())
print("nearest item vectors of", data.item_text[item] or data.item_ids[item])
lab.neighbours(model, data, item)

**Exercise.** Do one BPR update by hand with numpy, as in the optimisation primer: p_u = (0.1, 0.2), q_i = (0.3, 0.1), q_j = (0.2, 0.4), learning rate 0.1. Did x_uij grow?

<details><summary>Hint</summary>

x = p_u . (q_i - q_j); g = 1 / (1 + exp(x)); then the three updates.

</details>

<details><summary>Solution</summary>

```python
import numpy as np

p, qi, qj, lr = np.array([0.1, 0.2]), np.array([0.3, 0.1]), np.array([0.2, 0.4]), 0.1
x = p @ (qi - qj)
g = 1 / (1 + np.exp(x))
p2, qi2, qj2 = p + lr * g * (qi - qj), qi + lr * g * p, qj - lr * g * p
print(x, p2 @ (qi2 - qj2))  # -0.05 -> -0.04: the pair moved the right way
```

</details>

In [ ]:
# Your code here

## Level 2: understand each setting

For each sweep: **write your prediction in the cell below it first**, then run the sweep. Every other setting stays at the baseline's best. The lab page explains what to look for.

In [ ]:
table = lab.sweep(METHOD, DATASET, "bpr_lr", [0.001, 0.01, 0.05, 0.1], start="best")
lab.plot_sweep(table)
table

**Your prediction for `bpr_lr`, and what you saw:** ...

## Level 3: data tricks

Experiments run on the rented box over all five datasets, for example:

```bash
./scripts/run_lab_box.sh bpr_mf:more-iterations
```

Then fetch the results (`./scripts/fetch_lab_results.sh <host>`) and compare here. `more-iterations` is ready to run; `min-count`
needs a small code change first (the lab page has the hint and the folded solution). After writing it, run its test:
`pytest -q tests/test_bpr_mf_variant.py`.

In [ ]:
lab.compare(METHOD, f"{METHOD}:more-iterations")

In [ ]:
lab.compare(METHOD, f"{METHOD}:min-count")

## Level 4: one change from the literature

Write the variant (the lab page explains the idea, with hints and the folded solution), test it, uncomment
`popular-negatives` in `labs/09-bpr-mf/experiments.yaml`, run it on the box, fetch, then compare and look at the segments.

In [ ]:
lab.compare(METHOD, f"{METHOD}:popular-negatives")

In [ ]:
parts = lab.segments(METHOD, f"{METHOD}:popular-negatives", DATASET)
lab.plot_segments(parts, "activity")

In [ ]:
parts["items"]

## Record your results

Fill in the table at the end of the lab page, then rebuild the scoreboard: `python -m recbench.lab scoreboard --docs`.